# 10 · Baselines — Consumo urbano anual por municipio

Baselines de series temporales + **regla de negocio** (lo que un planificador de la DGRH haria hoy) para fijar la cota a superar.

- Split temporal: train <= 2021, test 2022-2024 (h = 3)
- Modelos: `naive`, `media`, `ets` (Holt amortiguado), `gb_temporal` (GB con lags/trend), `arima` (auto-ARIMA), `regla_negocio` (variacion del consumo = elasticidad fija 0,3 x variacion del IPH anual de la isla)
- Resultados: `results/10_baseline_metrics.csv`


In [ ]:
import polars as pl
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pmdarima import auto_arima
from sklearn.ensemble import GradientBoostingRegressor

sns.set_theme(style="whitegrid", palette="muted")
DATA = Path("data")
RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)

df = pl.read_csv(DATA / "abastecimiento_urbano_baleares.csv", infer_schema_length=None).select(["cod_municipio", "anio", "consumo_hm3"])
print("shape:", df.shape)


In [ ]:
TEST_START = 2022
H = 3

years = sorted(df["anio"].unique().to_list())
series = {
    int(cod[0]): s.sort("anio")["consumo_hm3"].to_list()
    for cod, s in df.partition_by("cod_municipio", as_dict=True).items()
}
print(f"{len(series)} municipios | anios {years[0]}..{years[-1]} | test {TEST_START}..{years[-1]}")

def split(s):
    train = np.array([v for y, v in zip(years, s) if y < TEST_START], float)
    test = np.array([v for y, v in zip(years, s) if y >= TEST_START], float)
    return train, test


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def metricas(test, pred):
    test, pred = np.asarray(test, float), np.asarray(pred, float)
    return dict(
        mae=float(mean_absolute_error(test, pred)),
        mape=float(np.mean(np.abs((test - pred) / test)) * 100),
        rmse=float(mean_squared_error(test, pred) ** 0.5),
        r2=float(r2_score(test, pred)),
    )


In [ ]:
# ── Baselines clasicos ────────────────────────────────────────────────────────
def naive_predict(train, h):
    return np.full(h, train[-1])

def media_predict(train, h):
    return np.full(h, train.mean())

def ets_predict(train, h):
    m = ExponentialSmoothing(train, trend="add", damped_trend=True, initialization_method="estimated")
    fit = m.fit(optimized=True, remove_bias=True)
    return np.asarray(fit.forecast(h), float)

def gb_predict(train, h):
    hist = list(train)
    preds = []
    for _ in range(h):
        X, y = [], []
        for i in range(3, len(hist)):
            X.append([i + 1, hist[i - 1], hist[i - 2], np.mean(hist[i - 3:i])])
            y.append(hist[i])
        if len(y) < 4:
            p = np.mean(hist)
        else:
            m = GradientBoostingRegressor(n_estimators=100, learning_rate=0.05, max_depth=2, random_state=42)
            m.fit(X, y)
            p = float(m.predict([[len(hist) + 1, hist[-1], hist[-2], np.mean(hist[-3:])]])[0])
        p = max(p, 0.0)
        preds.append(p)
        hist.append(p)
    return np.asarray(preds, float)

def arima_predict(train, h):
    m = auto_arima(train, seasonal=False, stepwise=True, d=1, max_p=2, max_q=2,
                   start_p=1, start_q=1, n_fits=10, suppress_warnings=True,
                   error_action="ignore", random_state=42)
    return np.asarray(m.predict(n_periods=h), float)


In [ ]:
# ── Regla de negocio (heuristica DGRH) ────────────────────────────────────────
# consumo(t) = consumo(t-1) x (1 + 0.3 x variacion interanual del IPH de la isla)
# Elasticidad fija 0.3: hipotesis de planificador (el consumo urbano sigue en un 30% la presion humana)
presion = pl.read_csv(DATA / "presion_humana.csv", infer_schema_length=None)
mun = pl.read_csv(DATA / "municipio.csv", infer_schema_length=None).select(["cod_municipio", "cod_provincia"])
isla_map = {"071": "Eivissa i Formentera", "072": "Eivissa i Formentera", "073": "Mallorca", "074": "Menorca"}
iph_anual = (presion.group_by(["nombre_isla", "anio"]).agg(pl.col("iph").mean())
             .sort(["nombre_isla", "anio"]))
iph_var = iph_anual.with_columns(
    (pl.col("iph").pct_change()).over("nombre_isla").alias("var_iph")
)
isla_map_bn = pl.DataFrame({"cod_provincia": [71, 72, 73, 74],
                        "isla": ["Eivissa i Formentera", "Eivissa i Formentera", "Mallorca", "Menorca"]})
mun_isla = mun.join(isla_map_bn, on="cod_provincia", how="left")

def factor_isla(cod):
    isla = mun_isla.filter(pl.col("cod_municipio") == cod)["isla"][0]
    filas = iph_var.filter((pl.col("nombre_isla") == isla) & pl.col("var_iph").is_not_null()).sort("anio")
    return {int(f["anio"]): float(f["var_iph"]) for f in filas.iter_rows(named=True)}

def regla_negocio_predict(cod, train, h):
    base_anio = years[len(train) - 1]
    var = factor_isla(cod)
    preds = []
    last = train[-1]
    for a in range(base_anio + 1, base_anio + h + 1):
        v = var.get(a)
        if v is None:
            v = 0.0
        last = max(last * (1 + 0.3 * v), 0.001)  # v = variacion interanual del IPH (fraccion)
        preds.append(last)
    return np.asarray(preds, float)


In [ ]:
# ── Ejecutar los 6 baselines ──────────────────────────────────────────────────
modelos = {
    "naive": naive_predict,
    "media": media_predict,
    "ets": ets_predict,
    "gb_temporal": gb_predict,
    "arima": arima_predict,
}
filas = []
for nombre, fn in modelos.items():
    for cod, s in series.items():
        train, test = split(s)
        m = metricas(test, fn(train, H))
        filas.append({"modelo": nombre, "cod_municipio": cod, **m})

# regla de negocio necesita el municipio
for cod, s in series.items():
    train, test = split(s)
    filas.append({"modelo": "regla_negocio", "cod_municipio": cod,
                  **metricas(test, regla_negocio_predict(cod, train, H))})

res = pd.DataFrame(filas)
res.to_csv(RESULTS / "10_baseline_metrics.csv", index=False)
print(res.groupby("modelo")[["mae", "mape", "rmse", "r2"]].mean().round(3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=res, x="modelo", y="mape", ax=axes[0])
axes[0].set_title("MAPE (%) por municipio y modelo")
axes[0].tick_params(axis="x", rotation=30)
sns.boxplot(data=res, x="modelo", y="mae", ax=axes[1])
axes[1].set_title("MAE (hm3) por municipio y modelo")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

best = res.loc[res.groupby("cod_municipio")["mape"].idxmin()]
print(best.groupby("modelo").size().rename("municipios ganados").sort_values(ascending=False))


**Conclusiones**

- `naive` suele ser la cota dura con series anuales cortas. `gb_temporal` es el puente hacia los modelos con features externas.
- `regla_negocio` es la referencia accionable: si el ML no la supera, la DGRH deberia seguir con su heuristica (mas barata).
